# 📊 IDX Exchange – EDA & Dataset Validation
### Weeks 2–3 | Data Analyst Internship
---
**Goal:** Inspect, validate, and enrich the combined residential MLS datasets before moving into cleaning and analytics.

**Steps covered in this notebook:**
1. Load datasets
2. Basic structure overview
3. Missing value analysis
4. Property type check
5. Numeric distribution review (ClosePrice, LivingArea, DaysOnMarket)
6. Mortgage rate enrichment (FRED API)
7. Save enriched datasets


## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Show all columns when printing DataFrames
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)


## 2. Load Datasets
Loading the combined sold and listings CSVs produced in Week 1.


In [ ]:
sold     = pd.read_csv("../output/sold_combined_residential.csv", low_memory=False)
listings = pd.read_csv("../output/listings_combined_residential.csv", low_memory=False)

print("✅ Sold rows & columns:    ", sold.shape)
print("✅ Listings rows & columns:", listings.shape)


## 3. Basic Structure Overview
Quick look at column names, data types, and the first few rows.


In [ ]:
print("=== SOLD: Column Names ===")
print(sold.columns.tolist())


In [ ]:
print("=== SOLD: Data Types ===")
print(sold.dtypes)


In [ ]:
sold.head(3)


## 4. Missing Value Analysis
Per the handbook: calculate missing counts and percentages, and flag any columns above **90% null**.


In [ ]:
missing_count   = sold.isnull().sum()
missing_pct     = (missing_count / len(sold)) * 100

missing_df = pd.DataFrame({
    "missing_count":   missing_count,
    "missing_percent": missing_pct
}).sort_values("missing_percent", ascending=False)

print("=== Top 20 columns with most missing values ===")
missing_df.head(20)


In [ ]:
# Flag columns above 90% missing — candidates to drop
high_missing = missing_df[missing_df["missing_percent"] > 90]

print(f"⚠️  Columns above 90% missing: {len(high_missing)}")
high_missing


## 5. Property Type Check
Verify the dataset is filtered to Residential only (done in Week 1, confirming here).


In [ ]:
print("=== PropertyType value counts ===")
print(sold["PropertyType"].value_counts())


## 6. Numeric Distribution Review
Handbook fields to analyze: **ClosePrice**, **LivingArea**, **DaysOnMarket**.

For each: descriptive stats + histogram + boxplot.


### 6.1 `ClosePrice`

In [ ]:
print("=== ClosePrice – Descriptive Stats ===")
print(sold["ClosePrice"].describe(percentiles=[.1, .25, .5, .75, .9]))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
sold["ClosePrice"].dropna().plot(kind="hist", bins=40, ax=ax1, color="#4C72B0", edgecolor="white")
ax1.set_title("ClosePrice – Distribution")
ax1.set_xlabel("Close Price ($)")
ax1.set_ylabel("Count")

# Boxplot
sold.boxplot(column="ClosePrice", ax=ax2, vert=False)
ax2.set_title("ClosePrice – Boxplot")
ax2.set_xlabel("Close Price ($)")

plt.tight_layout()
plt.show()


### 6.2 `LivingArea`

In [ ]:
print("=== LivingArea – Descriptive Stats ===")
print(sold["LivingArea"].describe(percentiles=[.1, .25, .5, .75, .9]))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
sold["LivingArea"].dropna().plot(kind="hist", bins=40, ax=ax1, color="#4C72B0", edgecolor="white")
ax1.set_title("LivingArea – Distribution")
ax1.set_xlabel("Living Area (sqft)")
ax1.set_ylabel("Count")

# Boxplot
sold.boxplot(column="LivingArea", ax=ax2, vert=False)
ax2.set_title("LivingArea – Boxplot")
ax2.set_xlabel("Living Area (sqft)")

plt.tight_layout()
plt.show()


### 6.3 `DaysOnMarket`

In [ ]:
print("=== DaysOnMarket – Descriptive Stats ===")
print(sold["DaysOnMarket"].describe(percentiles=[.1, .25, .5, .75, .9]))


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
sold["DaysOnMarket"].dropna().plot(kind="hist", bins=40, ax=ax1, color="#4C72B0", edgecolor="white")
ax1.set_title("DaysOnMarket – Distribution")
ax1.set_xlabel("Days on Market")
ax1.set_ylabel("Count")

# Boxplot
sold.boxplot(column="DaysOnMarket", ax=ax2, vert=False)
ax2.set_title("DaysOnMarket – Boxplot")
ax2.set_xlabel("Days on Market")

plt.tight_layout()
plt.show()


## 7. Mortgage Rate Enrichment (FRED API)
Fetch the 30-year fixed mortgage rate from FRED, resample to monthly averages, and merge onto both datasets.

> No API key needed — FRED provides a public CSV endpoint.


In [ ]:
# Step 1 – Fetch weekly mortgage rate data from FRED
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=["observation_date"])
mortgage.columns = ["date", "rate_30yr_fixed"]

print("✅ Mortgage data fetched:", mortgage.shape)
mortgage.tail(5)


In [ ]:
# Step 2 – Resample from weekly → monthly average
mortgage["year_month"] = mortgage["date"].dt.to_period("M")

mortgage_monthly = (
    mortgage
    .groupby("year_month")["rate_30yr_fixed"]
    .mean()
    .reset_index()
)

print("✅ Monthly averages computed:", mortgage_monthly.shape)
mortgage_monthly.tail(5)


In [ ]:
# Quick plot of mortgage rates over time
fig, ax = plt.subplots(figsize=(12, 4))
mortgage_monthly.set_index("year_month").plot(ax=ax, color="#C44E52", legend=False)
ax.set_title("30-Year Fixed Mortgage Rate (Monthly Average) – FRED")
ax.set_ylabel("Rate (%)")
ax.set_xlabel("")
plt.tight_layout()
plt.show()


In [ ]:
# Step 3 – Create year_month key on MLS datasets
sold["year_month"]     = pd.to_datetime(sold["CloseDate"]).dt.to_period("M")
listings["year_month"] = pd.to_datetime(listings["ListingContractDate"]).dt.to_period("M")


In [ ]:
# Step 4 – Merge mortgage rates onto both datasets
sold     = sold.merge(mortgage_monthly,     on="year_month", how="left")
listings = listings.merge(mortgage_monthly, on="year_month", how="left")


In [ ]:
# Step 5 – Validate: null rate count should be 0 (or very low for edge dates)
print("Sold – null rates:    ", sold["rate_30yr_fixed"].isnull().sum())
print("Listings – null rates:", listings["rate_30yr_fixed"].isnull().sum())

sold[["CloseDate", "year_month", "ClosePrice", "rate_30yr_fixed"]].head(5)


## 8. Save Enriched Datasets
Save both datasets with the mortgage rate column attached.


In [ ]:
sold.to_csv("../output/sold_with_rates.csv", index=False)
listings.to_csv("../output/listings_with_rates.csv", index=False)

print("✅ Saved: sold_with_rates.csv")
print("✅ Saved: listings_with_rates.csv")
print(f"   Sold final shape:     {sold.shape}")
print(f"   Listings final shape: {listings.shape}")


---
## ✅ Weeks 2–3 Complete

**What was done:**
- Loaded combined Residential MLS datasets
- Reviewed structure (columns, dtypes, sample rows)
- Identified missing values and flagged columns >90% null
- Confirmed PropertyType == Residential filter
- Analyzed distributions of ClosePrice, LivingArea, DaysOnMarket
- Fetched 30-yr mortgage rate from FRED and merged onto both datasets
- Saved enriched outputs: `sold_with_rates.csv` and `listings_with_rates.csv`

**Next → Weeks 4–5:** Data Cleaning & Preparation (date fields, invalid values, geographic checks)
